In [1]:
import copy
import csv
import json
import logging
import math
import random
import re
import shutil
import urllib.request
import warnings
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

warnings.filterwarnings("ignore")
# os.chdir(r"C:\你的專案資料夾\Data Training V2")

# ================================================================ 加入新資料時才改
REBUILD = True            # True = 先把舊的資料切分與 keyword_model 搬到備份資料夾，再用全部資料重新切分、訓練
                           #        跑完記得改回 False，否則每次執行都會重來

# ================================================================ 路徑設定
LABEL_DIR = Path(r"..\Data Preparation\Phase5-Pause\output\03labels")       # 03split_npy.py 的輸出
SEG_DIR = Path(r"..\Data Preparation\Phase5-Pause\output\03segments")     # 292 維 (含臉部)
DATA_DIR = Path(r"data")                     # 資料切分、字典、norm
OUT_DIR = Path(r"keyword_model")             # 這個 notebook 的所有輸出
JIEBA_DICT = DATA_DIR / "dict.txt.big"       # jieba 繁體詞典，找不到會自動下載
JIEBA_DICT_URL = "https://raw.githubusercontent.com/fxsjy/jieba/master/extra_dict/dict.txt.big"

# ================================================================ 選詞與篩選
MIN_WORD_COUNT = 30        # 訓練集出現至少幾次的詞才列為候選 (資料越多，達標的詞越多，詞彙清單會自動變大)
MAX_CANDIDATES = None      # 候選詞上限 (None = 不限；候選太多導致訓練太慢時可以設，例如 1000)
MIN_WORD_LEN = 2           # 至少幾個字
STOPWORDS = set()          # 想排除的詞，例如 {"表示", "已經"}
SELECT_MIN_VAL_AP = 0.1    # 第二階段只保留驗證集 AP 至少多少的詞
SELECT_MAX = None          # 最多保留幾個詞 (None = 學得起來的全部保留)
MIN_TEST_POSITIVE = 3      # 測試集至少出現幾次的詞才算進 mAP

# ================================================================ 模型與訓練
USE_FACE = False           # 臉部特徵在之前的實驗沒有幫助，預設只用手與 pose
STAGE1_CTC_WEIGHT = 0.0    # 第一階段不用 CTC (只用來篩選詞彙)
STAGE2_CTC_WEIGHT = 0.5    # 第二階段 (最終模型) CTC 的權重
ENSEMBLE_STAGE1 = True     # 最終分數 = 第二階段與第一階段模型的平均 (兩個都已訓練好，不用多花時間；
                           #   測試集上精確率 63% → 66%、每句答對的詞多約 28%)
D_MODEL = 256
N_HEADS = 4
ENC_LAYERS = 4
FF_DIM = 1024
DROPOUT = 0.3
BATCH_SIZE = 32
EPOCHS = 150
LR = 3e-4
WEIGHT_DECAY = 0.05
WARMUP_EPOCHS = 5
PATIENCE = 10              # val mAP 連續幾個 epoch 沒進步就停止
MAX_POS_WEIGHT = 10.0

# ================================================================ 展示
TARGET_PRECISION = 0.7     # 在驗證集上要求「輸出的詞有多少比例是對的」(測試集通常低幾個百分點)
MAX_WORDS = 5              # 「有把握」最多幾個詞
SHOW_WORDS = 5             # 兩層合計顯示幾個詞 (有把握的不夠時，用「其他可能」補滿)

# ================================================================ 資料 (跟之前的 notebook 相同，一般不用改)
SPLIT = (0.8, 0.1, 0.1)
SEED = 42
SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]
MIN_FREQ = 2
FEAT_DIM = 292
FLAG_DIMS = [126, 127, 155, 291]
USE_BLENDSHAPES = USE_LIPS = USE_HEAD_POSE = USE_FACE
FRAME_STRIDE = 2
MAX_FRAMES = 300
MAX_TEXT_LEN = 64
SPEED_RANGE = (0.8, 1.2)
NOISE_STD = 0.05
NORM_FILE = "norm_masked_face.npz"
AUGMENT = True
ROTATE_DEG = 10
SCALE_RANGE = (0.9, 1.1)
SHIFT = 0.05
TIME_MASK_N = 2
TIME_MASK_MAX = 0.1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
PAD, BOS, EOS, UNK = 0, 1, 2, 3
print("device:", DEVICE)

device: cuda


## 0. 準備資料
檢查有沒有還沒用到的新影片；REBUILD = True 時先備份舊的資料和模型。接著依影片分成訓練、驗證、測試集（8:1:1），計算標準化參數

In [2]:
SPLIT_FILES = ["train.csv", "val.csv", "test.csv", "vocab.json", "norm.npz", NORM_FILE]

if REBUILD:
    backup = Path(f"backup_{datetime.now():%Y%m%d_%H%M%S}")
    moved = []
    for name in SPLIT_FILES:
        if (DATA_DIR / name).exists():
            (backup / DATA_DIR).mkdir(parents=True, exist_ok=True)
            shutil.move(str(DATA_DIR / name), str(backup / DATA_DIR / name))
            moved.append(str(DATA_DIR / name))
    if OUT_DIR.exists():
        backup.mkdir(parents=True, exist_ok=True)
        shutil.move(str(OUT_DIR), str(backup / OUT_DIR.name))
        moved.append(str(OUT_DIR))
    print(f"REBUILD：已把 {len(moved)} 項搬到 {backup}：{'、'.join(moved) or '(沒有東西要搬)'}")
    print("這次會用全部資料重新切分、訓練；跑完記得把 REBUILD 改回 False")

if (DATA_DIR / "train.csv").exists():
    in_split = set()
    for split in ("train", "val", "test"):
        with (DATA_DIR / f"{split}.csv").open(encoding="utf-8-sig", newline="") as f:
            in_split |= {row["path"].rsplit("/", 1)[0] for row in csv.DictReader(f)}
    available = {txt.parent.relative_to(LABEL_DIR).as_posix() for txt in LABEL_DIR.glob("*/*/*.txt")
                 if txt.read_text(encoding="utf-8").strip()}
    new_videos = sorted(available - in_split)
    if new_videos:
        parts = Counter(v.split("/")[0] for v in new_videos)
        print(f"[注意] {LABEL_DIR} 有 {len(new_videos)} 支影片不在目前的資料切分中 "
              f"(各 part：{dict(sorted(parts.items()))})")
        print("       目前會沿用舊的資料切分，這些影片不會被用到。要加進來請把 REBUILD 改成 True 再執行")
    else:
        print(f"資料切分涵蓋 {LABEL_DIR} 裡所有 {len(available)} 支影片")

REBUILD：已把 7 項搬到 backup_20261123_155303：data\train.csv、data\val.csv、data\test.csv、data\vocab.json、data\norm.npz、data\norm_masked_face.npz、keyword_model
這次會用全部資料重新切分、訓練；跑完記得把 REBUILD 改回 False


In [3]:
if all((DATA_DIR / name).exists() for name in ("train.csv", "val.csv", "test.csv", "vocab.json", "norm.npz")):
    print(f"沿用 {DATA_DIR} 裡現有的資料切分、字典與 norm")
else:
    print(f"{DATA_DIR} 沒有完整資料，重新建立")
    groups = {}
    for txt in sorted(LABEL_DIR.glob("*/*/*.txt")):
        group = txt.parent.relative_to(LABEL_DIR)
        for line in txt.read_text(encoding="utf-8").splitlines():
            name, _, text = line.partition(": ")
            if text.strip():
                groups.setdefault(group.as_posix(), []).append(((group / f"{name}.npy").as_posix(), text.strip()))

    keys = sorted(groups)
    random.Random(SEED).shuffle(keys)
    n_train, n_val = round(len(keys) * SPLIT[0]), round(len(keys) * SPLIT[1])
    splits = {"train": keys[:n_train], "val": keys[n_train:n_train + n_val], "test": keys[n_train + n_val:]}

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    rows = {}
    for split, split_keys in splits.items():
        rows[split] = [row for key in sorted(split_keys) for row in groups[key]]
        with (DATA_DIR / f"{split}.csv").open("w", newline="", encoding="utf-8-sig") as f:
            writer = csv.writer(f)
            writer.writerow(("path", "text"))
            writer.writerows(rows[split])

    freq = Counter(c for _, text in rows["train"] for c in text)
    chars = sorted(c for c, n in freq.items() if n >= MIN_FREQ)
    (DATA_DIR / "vocab.json").write_text(json.dumps(SPECIALS + chars, ensure_ascii=False), encoding="utf-8")

    total, total_sq, count = np.zeros(FEAT_DIM), np.zeros(FEAT_DIM), 0
    for path, _ in tqdm(rows["train"], desc="norm"):
        arr = np.load(SEG_DIR / path).astype(np.float64)
        total += arr.sum(0)
        total_sq += (arr ** 2).sum(0)
        count += len(arr)
    mean = total / count
    std = np.sqrt(np.maximum(total_sq / count - mean ** 2, 0)) + 1e-6
    mean[FLAG_DIMS], std[FLAG_DIMS] = 0.0, 1.0
    np.savez(DATA_DIR / "norm.npz", mean=mean.astype(np.float32), std=std.astype(np.float32))

for split in ("train", "val", "test"):
    with (DATA_DIR / f"{split}.csv").open(encoding="utf-8-sig", newline="") as f:
        print(f"{split}: {sum(1 for _ in csv.DictReader(f))} 句")
print("字元數:", len(json.loads((DATA_DIR / "vocab.json").read_text(encoding="utf-8"))) - len(SPECIALS))

# ---------------------------------------------------------------- 特徵位置與缺漏遮罩
LEFT, RIGHT, POSE = slice(0, 63), slice(63, 126), slice(128, 155)
LEFT_FLAG, RIGHT_FLAG, POSE_FLAG = 126, 127, 155
BLENDSHAPES, LIPS, HEAD_POSE, FACE_FLAG = slice(156, 208), slice(208, 288), slice(288, 291), 291


def valid_mask(arr):
    """(T, FEAT_DIM) 的 bool：每一維是不是真的有偵測到。旗標本身永遠算有效。"""
    m = np.ones(arr.shape, dtype=bool)
    m[:, LEFT] = arr[:, [LEFT_FLAG]] > 0.5
    m[:, RIGHT] = arr[:, [RIGHT_FLAG]] > 0.5
    point_ok = (arr[:, POSE].reshape(-1, 9, 3)[:, :, 2] > 0) & (arr[:, [POSE_FLAG]] > 0.5)   # 信心值 0 = 這個點沒偵測到
    m[:, POSE] = np.repeat(point_ok, 3, axis=1)
    m[:, 156:FACE_FLAG] = arr[:, [FACE_FLAG]] > 0.5            # 沒偵測到臉 → 臉部特徵全部無效
    for part, use in ((BLENDSHAPES, USE_BLENDSHAPES), (LIPS, USE_LIPS), (HEAD_POSE, USE_HEAD_POSE)):
        if not use:
            m[:, part] = False                                   # 關掉的特徵組一律設成 0
    if not (USE_BLENDSHAPES or USE_LIPS or USE_HEAD_POSE):
        m[:, FACE_FLAG] = False                                  # 完全不用臉時，連「有沒有臉」也不給
    return m


# ---------------------------------------------------------------- 先確認每一句的分段檔都存在
missing = Counter()
for split in ("train", "val", "test"):
    with (DATA_DIR / f"{split}.csv").open(encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f):
            if not (SEG_DIR / row["path"]).exists():
                missing[row["path"].split("/")[0]] += 1
if missing:
    print(f"[錯誤] {SEG_DIR} 缺少 {sum(missing.values())} 個分段檔，各 part 缺少的數量：")
    for part, n in sorted(missing.items(), key=lambda x: (len(x[0]), x[0])):
        print(f"  part {part}：{n} 個")
    print("請依序確認缺少的 part：")
    print("  1. Phase4-Keypoint/Keypoint_output/01face/<part> 有沒有檔案 → 沒有的話跑 02keypoint_face.py")
    print("  2. Phase4-Keypoint/Keypoint_output/02combined/<part> 有沒有檔案、是不是 292 維 → 沒有的話跑 03keypoint_combine.py")
    print("  3. 03split_npy.py，part 輸入 all 重跑")
    print("  也檢查 Phase4-Keypoint/、Phase5-Pause/ 裡有沒有 failed_*_part_<part>.txt 失敗紀錄")
    raise FileNotFoundError(f"{SEG_DIR} 缺少分段檔，見上方說明")
print(f"{SEG_DIR} 的分段檔都齊全")

if (DATA_DIR / NORM_FILE).exists():
    print(f"沿用 {DATA_DIR / NORM_FILE}")
else:
    with (DATA_DIR / "train.csv").open(encoding="utf-8-sig", newline="") as f:
        train_paths = [row["path"] for row in csv.DictReader(f)]
    total, total_sq, count = np.zeros(FEAT_DIM), np.zeros(FEAT_DIM), np.zeros(FEAT_DIM)
    for path in tqdm(train_paths, desc="masked norm"):
        arr = np.load(SEG_DIR / path).astype(np.float64)
        m = valid_mask(arr)
        arr = np.where(m, arr, 0.0)
        total += arr.sum(0)
        total_sq += (arr ** 2).sum(0)
        count += m.sum(0)
    mean = total / np.maximum(count, 1)
    std = np.sqrt(np.maximum(total_sq / np.maximum(count, 1) - mean ** 2, 0)) + 1e-6
    mean[FLAG_DIMS], std[FLAG_DIMS] = 0.0, 1.0
    np.savez(DATA_DIR / NORM_FILE, mean=mean.astype(np.float32), std=std.astype(np.float32),
             valid_ratio=(count / count.max()).astype(np.float32),
             flag_rate=(total / np.maximum(count, 1))[FLAG_DIMS].astype(np.float32))   # 左手、右手、pose、臉的偵測率
    print(f"已存成 {DATA_DIR / NORM_FILE}")

ratio = np.load(DATA_DIR / NORM_FILE)["valid_ratio"]
print(f"訓練集偵測率：左手 {ratio[0]:.1%}、右手 {ratio[63]:.1%}、左腕 {ratio[128 + 3 * 5]:.1%}、右腕 {ratio[128 + 3 * 6]:.1%}、"
      f"臉 {np.load(DATA_DIR / NORM_FILE)['flag_rate'][-1]:.1%}")
print(f"臉部特徵：表情分數 {'開' if USE_BLENDSHAPES else '關'}、嘴唇 {'開' if USE_LIPS else '關'}、頭部轉向 {'開' if USE_HEAD_POSE else '關'}")
with (DATA_DIR / "train.csv").open(encoding="utf-8-sig", newline="") as f:
    first = next(csv.DictReader(f))["path"]
dim = np.load(SEG_DIR / first, mmap_mode="r").shape[1]
assert dim == FEAT_DIM, f"{SEG_DIR / first} 是 {dim} 維，應該是 {FEAT_DIM} 維：請確認 Phase4/5 已經用臉部版本重新產生"

data 沒有完整資料，重新建立


norm: 100%|██████████| 25276/25276 [02:54<00:00, 144.81it/s]


train: 25276 句
val: 3036 句
test: 2775 句
字元數: 2813
..\Data Preparation\Phase5-Pause\output\03segments_face 的分段檔都齊全


masked norm: 100%|██████████| 25276/25276 [00:06<00:00, 3949.63it/s]

已存成 data\norm_masked_face.npz
訓練集偵測率：左手 70.2%、右手 93.4%、左腕 99.3%、右腕 99.6%、臉 0.0%
臉部特徵：表情分數 關、嘴唇 關、頭部轉向 關


In [4]:
def load_vocab():
    return json.loads((DATA_DIR / "vocab.json").read_text(encoding="utf-8"))


def augment_geometry(x):
    """隨機旋轉 (pose 與手形一起轉)、pose 縮放與平移。輸入是還沒標準化的原始座標。"""
    x = x.copy()
    T = len(x)
    angle = np.deg2rad(np.random.uniform(-ROTATE_DEG, ROTATE_DEG))
    c, s = np.cos(angle), np.sin(angle)
    rot = np.array([[c, -s], [s, c]], dtype=np.float32)

    pose = x[:, POSE].reshape(T, 9, 3)
    ok = pose[:, :, 2:] > 0
    xy = (pose[:, :, :2] - 0.5) @ rot.T * np.random.uniform(*SCALE_RANGE) + 0.5 \
        + np.random.uniform(-SHIFT, SHIFT, 2).astype(np.float32)
    pose[:, :, :2] = np.where(ok, xy, 0.0)
    x[:, POSE] = pose.reshape(T, -1)

    hands = x[:, :126].reshape(T, 2, 21, 3)          # 手形已經以手腕為原點，直接轉 (沒偵測到的是 0，轉完還是 0)
    hands[..., :2] = hands[..., :2] @ rot.T
    x[:, :126] = hands.reshape(T, -1)
    return x


def time_mask(x):
    """隨機把幾段連續的格整段設成 0 (含旗標)，模擬遮擋或漏偵測。"""
    T = len(x)
    for _ in range(TIME_MASK_N):
        width = np.random.randint(0, max(1, int(T * TIME_MASK_MAX)) + 1)
        if width:
            start = np.random.randint(0, T - width + 1)
            x[start:start + width] = 0.0
    return x


class SignDataset(Dataset):
    def __init__(self, split, vocab, train=False):
        with (DATA_DIR / f"{split}.csv").open(encoding="utf-8-sig", newline="") as f:
            self.rows = [(row["path"], row["text"]) for row in csv.DictReader(f)]
        self.stoi = {c: i for i, c in enumerate(vocab)}
        norm = np.load(DATA_DIR / NORM_FILE)
        self.mean, self.std = norm["mean"], norm["std"]
        self.train = train
        # 快取「原始值」(只做取樣)，資料增強要在標準化之前做
        self.cache = [
            np.ascontiguousarray(np.load(SEG_DIR / path).astype(np.float32)[::FRAME_STRIDE])
            for path, _ in tqdm(self.rows, desc=f"load {split}")
        ]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        text = self.rows[i][1]
        raw = self.cache[i]
        if self.train:
            n = max(1, round(len(raw) * random.uniform(*SPEED_RANGE)))
            raw = raw[np.linspace(0, len(raw) - 1, n).round().astype(int)]
            if AUGMENT:
                raw = augment_geometry(raw)

        mask = valid_mask(raw)
        x = np.where(mask, (raw - self.mean) / self.std, 0.0).astype(np.float32)   # 沒偵測到的一律 0

        if self.train:
            noise = np.random.normal(0, NOISE_STD, x.shape).astype(np.float32)
            noise[:, FLAG_DIMS] = 0.0
            x = x + noise * mask                       # 雜訊只加在有偵測到的座標
            if AUGMENT:
                x = time_mask(x)
        x = x[:MAX_FRAMES]
        ids = [BOS] + [self.stoi.get(c, UNK) for c in text[:MAX_TEXT_LEN - 2]] + [EOS]
        return torch.from_numpy(np.ascontiguousarray(x)), torch.tensor(ids)


def collate(batch):
    xs, ys = zip(*batch)
    src = nn.utils.rnn.pad_sequence(xs, batch_first=True)
    lengths = torch.tensor([len(x) for x in xs])
    src_mask = torch.arange(src.size(1))[None, :] >= lengths[:, None]     # True = padding
    tgt = nn.utils.rnn.pad_sequence(ys, batch_first=True, padding_value=PAD)
    return src, src_mask, tgt

## 1. 選詞
用 jieba 斷詞，訓練集出現 ≥ 30 次的詞都列為候選

In [5]:
import jieba

jieba.setLogLevel(logging.WARNING)
if not JIEBA_DICT.exists():
    try:
        print(f"下載 jieba 繁體詞典到 {JIEBA_DICT} ...")
        urllib.request.urlretrieve(JIEBA_DICT_URL, JIEBA_DICT)
    except Exception as exc:
        print(f"[注意] 下載失敗 ({exc})，改用 jieba 預設詞典；繁體斷詞會比較不準")
if JIEBA_DICT.exists():
    jieba.set_dictionary(str(JIEBA_DICT))
jieba.initialize()

WORD_RE = re.compile(r"[\u4e00-\u9fff]+")
_token_cache = {}


def tokens(text):
    if text not in _token_cache:
        _token_cache[text] = jieba.lcut(text)
    return _token_cache[text]


def words_of(text):
    """一句字幕裡的詞 (去重複)，只留中文、長度 >= MIN_WORD_LEN、不在 STOPWORDS 的。"""
    return {w for w in tokens(text) if len(w) >= MIN_WORD_LEN and WORD_RE.fullmatch(w) and w not in STOPWORDS}


def read_rows(split):
    with (DATA_DIR / f"{split}.csv").open(encoding="utf-8-sig", newline="") as f:
        return [(row["path"], row["text"]) for row in csv.DictReader(f)]


train_rows, val_rows, test_rows = read_rows("train"), read_rows("val"), read_rows("test")
df = Counter(w for _, text in train_rows for w in words_of(text))
# 次數相同時依詞本身排序，確保每次執行選出的詞與順序完全相同 (中斷後重跑才能沿用之前的結果)
ranked = [w for w, c in sorted(df.items(), key=lambda x: (-x[1], x[0])) if c >= MIN_WORD_COUNT]
STAGE1_KEYWORDS = ranked[:MAX_CANDIDATES] if MAX_CANDIDATES else ranked

(OUT_DIR / "stage1").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "stage1" / "keywords.json").write_text(
    json.dumps({w: df[w] for w in STAGE1_KEYWORDS}, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"出現 >= {MIN_WORD_COUNT} 次的候選詞：{len(STAGE1_KEYWORDS)} 個 "
      f"(訓練集出現次數 {df[STAGE1_KEYWORDS[0]]} ~ {df[STAGE1_KEYWORDS[-1]]} 句)")
print("前 50 個：", "、".join(STAGE1_KEYWORDS[:50]))

出現 >= 30 次的候選詞：695 個 (訓練集出現次數 1049 ~ 30 句)
前 50 個： 昨天、台灣、表示、美國、已經、關心、來看、時間、各地、不過、報導、總統、中國、沒有、民眾、發生、今天、今年、自由、提供、目前、宣布、中國時報、時報、聯合報、讀報、接著、因為、提到、氣象、天氣、以及、發現、可以、影響、地區、繼續、超過、造成、出現、可能、公布、資料、政府、重點、如果、進行、另外、颱風、調查


## 2. 第一階段訓練
用所有候選詞訓練一次，判斷哪些詞學得起來（這個模型在第 4 步也會一起使用）

In [6]:
class KeywordDataset(Dataset):
    """共用 SignDataset 已經讀好的資料 (兩個階段不用重讀)，答案換成指定的關鍵詞：
    labels = 每個詞有沒有出現 (0/1)；seqs = 詞依出現順序的編號 (CTC 用，0 留給 blank)。"""

    def __init__(self, base, keywords):
        self.base, self.rows = base, base.rows
        index = {w: i for i, w in enumerate(keywords)}
        self.labels = np.zeros((len(base.rows), len(keywords)), dtype=np.float32)
        self.seqs = []
        for i, (_, text) in enumerate(base.rows):
            seq = [index[w] for w in tokens(text) if w in index]
            self.labels[i, seq] = 1.0
            self.seqs.append([j + 1 for j in seq])

    def __len__(self):
        return len(self.base)

    def __getitem__(self, i):
        x, _ = self.base[i]
        return x, torch.from_numpy(self.labels[i]), torch.tensor(self.seqs[i], dtype=torch.long)


def collate_kw(batch):
    xs, ys, seqs = zip(*batch)
    src = nn.utils.rnn.pad_sequence(xs, batch_first=True)
    lengths = torch.tensor([len(x) for x in xs])
    src_mask = torch.arange(src.size(1))[None, :] >= lengths[:, None]     # True = padding
    targets = torch.cat(seqs) if any(len(s) for s in seqs) else torch.zeros(0, dtype=torch.long)
    return src, src_mask, torch.stack(ys), targets, torch.tensor([len(s) for s in seqs])


vocab = load_vocab()
base_train = SignDataset("train", vocab, train=True)
base_val = SignDataset("val", vocab)
base_test = SignDataset("test", vocab)
print("臉部特徵：", "使用" if USE_FACE else "不使用 (全部遮掉)")

load train:   0%|          | 0/25276 [00:00<?, ?it/s]

load test: 100%|██████████| 2775/2775 [00:18<00:00, 150.11it/s]

臉部特徵： 不使用 (全部遮掉)


In [7]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2000):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:x.size(1)]


class SignKeywordCTC(nn.Module):
    """encoder 跟上一版相同。
    分類層：整段特徵 (平均 + 注意力加權) → 每個關鍵詞一個分數
    CTC 層：每一格 → 「blank 或哪個關鍵詞」"""

    def __init__(self, n_keywords, feat_dim=FEAT_DIM):
        super().__init__()
        self.src_proj = nn.Sequential(
            nn.Linear(feat_dim, D_MODEL), nn.LayerNorm(D_MODEL), nn.GELU(),
            nn.Dropout(DROPOUT), nn.Linear(D_MODEL, D_MODEL),
        )
        self.pos = PositionalEncoding(D_MODEL)
        layer = nn.TransformerEncoderLayer(D_MODEL, N_HEADS, FF_DIM, DROPOUT, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, ENC_LAYERS, norm=nn.LayerNorm(D_MODEL))
        self.attn = nn.Linear(D_MODEL, 1)
        self.head = nn.Sequential(nn.LayerNorm(2 * D_MODEL), nn.Dropout(DROPOUT), nn.Linear(2 * D_MODEL, n_keywords))
        self.ctc_head = nn.Linear(D_MODEL, n_keywords + 1)

    def forward(self, src, src_mask):
        h = self.encoder(self.pos(self.src_proj(src)), src_key_padding_mask=src_mask)
        valid = (~src_mask).unsqueeze(-1).float()
        mean = (h * valid).sum(1) / valid.sum(1).clamp(min=1)
        w = self.attn(h).squeeze(-1).float().masked_fill(src_mask, float("-inf")).softmax(-1)
        attended = (w.unsqueeze(-1) * h.float()).sum(1)
        logits = self.head(torch.cat([mean.float(), attended], dim=-1))
        return logits, self.ctc_head(h).float()

In [8]:
def average_precision(y, s):
    """一個詞的 AP：依分數由高到低排序，每找到一個正確的句子就記下當時的精確率，再取平均。"""
    if y.sum() == 0:
        return float("nan")
    order = np.argsort(-s, kind="stable")
    y = y[order]
    precision = np.cumsum(y) / np.arange(1, len(y) + 1)
    return float((precision * y).sum() / y.sum())


def per_word_ap(labels, scores):
    return np.array([average_precision(labels[:, j], scores[:, j]) for j in range(labels.shape[1])])


def eval_words(labels):
    """測試集出現至少 MIN_TEST_POSITIVE 次的詞才算進 mAP。"""
    return labels.sum(0) >= MIN_TEST_POSITIVE


def mean_ap(labels, scores, mask=None):
    ap = per_word_ap(labels, scores)
    mask = eval_words(labels) if mask is None else mask
    return float(np.nanmean(ap[mask]))


def shuffle_within(scores, groups, seed=SEED):
    """把預測在同一組 (同一則新聞) 的句子之間打亂。"""
    rng = np.random.default_rng(seed)
    out = scores.copy()
    by_group = defaultdict(list)
    for i, g in enumerate(groups):
        by_group[g].append(i)
    for idx in by_group.values():
        out[idx] = scores[rng.permutation(idx)]
    return out


def best_thresholds(labels, scores):
    """每個詞在驗證集上挑一個讓 F1 最高的門檻。"""
    out = np.full(labels.shape[1], 0.5)
    for j in range(labels.shape[1]):
        y, s = labels[:, j], scores[:, j]
        if y.sum() == 0:
            continue
        order = np.argsort(-s)
        tp = np.cumsum(y[order])
        f1 = 2 * tp / (np.arange(1, len(y) + 1) + y.sum())
        k = int(f1.argmax())
        out[j] = s[order[k]]
    return out


def micro_f1(labels, pred):
    tp = (labels * pred).sum()
    return float(2 * tp / max(1.0, labels.sum() + pred.sum()))


@torch.no_grad()
def predict_scores(model, loader, keep_ctc=False):
    """回傳 (分類層機率, CTC 偵測分數[, 每一格的 CTC 機率])。"""
    model.eval()
    cls, det, frames = [], [], []
    for src, src_mask, *_ in loader:
        src, src_mask = src.to(DEVICE), src_mask.to(DEVICE)
        with torch.autocast(DEVICE, enabled=USE_AMP):
            logits, ctc_logits = model(src, src_mask)
        prob = ctc_logits.softmax(-1).masked_fill(src_mask.unsqueeze(-1), 0.0)
        cls.append(torch.sigmoid(logits.float()).cpu().numpy())
        det.append(prob[:, :, 1:].amax(1).cpu().numpy())
        if keep_ctc:
            lengths = (~src_mask).sum(1).tolist()
            frames += [p[:n].cpu().numpy() for p, n in zip(prob, lengths)]
    return (np.concatenate(cls), np.concatenate(det)) + ((frames,) if keep_ctc else ())

In [9]:
def train_stage(name, keywords, ctc_weight):
    """訓練一個階段；已經跑過 (而且詞彙相同) 就直接讀結果。回傳 (資料集們, scores)。"""
    out = OUT_DIR / name
    ds = {s: KeywordDataset(b, keywords) for s, b in (("train", base_train), ("val", base_val), ("test", base_test))}
    done = (out / "scores.npz").exists() and (out / "best.pt").exists() and (out / "run_info.json").exists()
    if done and json.loads((out / "run_info.json").read_text(encoding="utf-8")).get("keywords") == keywords:
        print(f"{name}：已經跑過，沿用 {out}")
        return ds, np.load(out / "scores.npz")

    out.mkdir(parents=True, exist_ok=True)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    train_loader = DataLoader(ds["train"], BATCH_SIZE, shuffle=True, collate_fn=collate_kw)
    val_loader = DataLoader(ds["val"], BATCH_SIZE, collate_fn=collate_kw)
    test_loader = DataLoader(ds["test"], BATCH_SIZE, collate_fn=collate_kw)

    model = SignKeywordCTC(len(keywords)).to(DEVICE)
    pos = ds["train"].labels.sum(0)
    pos_weight = torch.tensor(np.clip(np.sqrt((len(ds["train"]) - pos) / np.maximum(pos, 1)), 1, MAX_POS_WEIGHT),
                              dtype=torch.float32, device=DEVICE)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    ctc = nn.CTCLoss(blank=0, zero_infinity=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps, warmup = EPOCHS * len(train_loader), WARMUP_EPOCHS * len(train_loader)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lambda step: step / warmup if step < warmup
        else 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total_steps - warmup))),
    )
    scaler = torch.amp.GradScaler(enabled=USE_AMP)

    history, best_map, best_epoch, stale = [], -1.0, 0, 0
    bar = tqdm(range(1, EPOCHS + 1), desc=name)
    for epoch in bar:
        model.train()
        total, n = 0.0, 0
        for src, src_mask, y, targets, target_lengths in train_loader:
            src, src_mask, y = src.to(DEVICE), src_mask.to(DEVICE), y.to(DEVICE)
            with torch.autocast(DEVICE, enabled=USE_AMP):
                logits, ctc_logits = model(src, src_mask)
                loss = bce(logits.float(), y)
                if ctc_weight > 0:
                    log_probs = ctc_logits.log_softmax(-1).transpose(0, 1)        # (T, N, C)
                    loss = loss + ctc_weight * ctc(log_probs, targets.to(DEVICE), (~src_mask).sum(1),
                                                   target_lengths.to(DEVICE))
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total, n = total + loss.item() * len(y), n + len(y)

        val_cls, _ = predict_scores(model, val_loader)
        val_map = mean_ap(ds["val"].labels, val_cls)
        history.append((total / n, val_map))
        if val_map > best_map:
            best_map, best_epoch, stale = val_map, epoch, 0
            torch.save(model.state_dict(), out / "best.pt")
        else:
            stale += 1
        bar.set_postfix(loss=f"{total / n:.4f}", val_mAP=f"{val_map:.3f}", best=f"{best_map:.3f}")
        if stale >= PATIENCE:
            break

    model.load_state_dict(torch.load(out / "best.pt", map_location=DEVICE))
    val_cls, val_det = predict_scores(model, val_loader)
    test_cls, test_det = predict_scores(model, test_loader)
    np.savez(out / "scores.npz", val=val_cls, test=test_cls, val_ctc=val_det, test_ctc=test_det)
    (out / "run_info.json").write_text(json.dumps(
        {"keywords": keywords, "ctc_weight": ctc_weight, "use_face": USE_FACE, "best_epoch": best_epoch,
         "best_val_map": best_map, "history": history}, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"{name} 完成：最佳 epoch {best_epoch}，val mAP {best_map:.3f}")
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return ds, np.load(out / "scores.npz")

In [10]:
stage1_ds, stage1_scores = train_stage("stage1", STAGE1_KEYWORDS, STAGE1_CTC_WEIGHT)
print(f"第一階段：測試集 mAP {mean_ap(stage1_ds['test'].labels, stage1_scores['test']):.3f} ({len(STAGE1_KEYWORDS)} 個詞)")

stage1:  40%|████      | 60/150 [24:52<37:18, 24.88s/it, best=0.157, loss=0.0142, val_mAP=0.157] 


stage1 完成：最佳 epoch 51，val mAP 0.157
第一階段：測試集 mAP 0.188 (695 個詞)


## 3. 篩選詞彙
只保留驗證集 AP ≥ 0.1 的詞

In [11]:
val_ap = per_word_ap(stage1_ds["val"].labels, stage1_scores["val"])
order = [j for j in np.argsort(-np.nan_to_num(val_ap, nan=-1), kind="stable") if val_ap[j] >= SELECT_MIN_VAL_AP][:SELECT_MAX or None]
KEYWORDS = [STAGE1_KEYWORDS[j] for j in order]
(OUT_DIR / "keywords.json").write_text(
    json.dumps({STAGE1_KEYWORDS[j]: round(float(val_ap[j]), 4) for j in order}, ensure_ascii=False, indent=1),
    encoding="utf-8")
covered = sum(bool(words_of(t) & set(KEYWORDS)) for _, t in train_rows) / len(train_rows)
print(f"{len(STAGE1_KEYWORDS)} 個詞 → 保留 {len(KEYWORDS)} 個；{covered:.0%} 的訓練句子至少含一個")
print("保留的詞 (依驗證集 AP 排序)：", "、".join(KEYWORDS))
dropped = [STAGE1_KEYWORDS[j] for j in np.argsort(np.nan_to_num(val_ap, nan=-1))[:20]]
print("刪掉的詞 (AP 最低的 20 個)：", "、".join(dropped))

695 個詞 → 保留 285 個；61% 的訓練句子至少含一個
保留的詞 (依驗證集 AP 排序)： 詳細資料、高度、陣雨、馬祖、讀報、接著、短暫、雷陣雨、氣象、報紙、昨天下午、澎湖、捷運、委員會、多雲、局部、屏東、時間、各地、資料、指揮中心、來看、重點、雷雨、關心、權益、聯合報、金門、提到、零星、半島、東北部、明天、董事長、天氣、繼續、恆春、衛生局、柯文、我們、交保、時報、颱風、地區、電價、季風、東北、侯友宜、名單、拍板、中部、降雨、條例、法官、立委、兩岸、修法、本土、報告、其他、學者、立法院、山區、自由、川普、交通部、行政院長、羈押、報導、提供、今天、機率、中南部、陸委會、中國時報、高雄、市長、這是、大多、第一、部長、東部、主席、大雨、新聞、產品、市府、預計、新增、印度、以下、則是、身心、發布、縣市、公布、北部、雲林、規模、開罰、準備、凌晨、不同、符合、世界、晚間、萬安、支持、指控、千元、日本、台北、死亡、疫情、交通、價格、卓榮泰、台南市、這次、國民黨、午後、評估、病例、去年、清晨、今年、食藥署、台南、明年、億元、東半、民進黨、決定、修正、送醫、台北市、影響、參選人、做好、教育、方案、進口、行政院、流感、治療、宜蘭、來自、通過、宣布、裁定、衛福部、上路、是否、最快、以北、撤離、需要、受傷、嫌犯、口罩、知名、萬元、新冠、說明、上週、昨天、賴清德、南韓、結果、只要、加上、地檢署、設置、消費者、天將、引起、代表、連續、參與、質疑、最近、總統、集團、爭議、措施、基隆、搜索、以來、希望、開放、接受、小時、即將、伊朗、台電、受到、月底、面臨、草案、外界、學校、結束、當時、全球、留意、勞動部、重大、恐怕、最重、員工、安全、強調、中國、新北、最後、造成、教育部、執行、無法、過去、召開、政治、確認、入境、奧運、美國、擔心、釐清、違規、統計、台灣、不到、團體、桃園、甚至、醫師、調漲、新竹、疫苗、就是、豪雨、記者會、調整、獲得、學生、緊急、烏克蘭、立即、帶來、疑慮、重要、紀錄、升溫、申請、呼籲、抗議、大量、起訴、事件、仍然、醫院、至於、發現、不滿、疑似、表示、居民、協會、近期、預估、必須、增加、國家、高溫、最高、舉辦、駕駛、南部、一起、服務、發生、大選、下午、氣溫、內政部
刪掉的詞 (AP 最低的 20 個)： 吸引、認定、外交部、每年、幼兒園、停課、長達、禁止、全案、性別、組織、

## 4. 第二階段訓練
用篩選後的詞加上 CTC 輔助損失訓練最終模型，最終分數是兩個階段模型分數的平均

In [12]:
ds, scores = train_stage("stage2", KEYWORDS, STAGE2_CTC_WEIGHT)
val_labels, test_labels = ds["val"].labels, ds["test"].labels
stage2_val = (scores["val"] + scores["val_ctc"]) / 2 if STAGE2_CTC_WEIGHT > 0 else scores["val"]
stage2_test = (scores["test"] + scores["test_ctc"]) / 2 if STAGE2_CTC_WEIGHT > 0 else scores["test"]
STAGE1_COLUMNS = list(map(int, order))         # 最終詞彙在第一階段輸出中的位置 (第 3 步算出)
if ENSEMBLE_STAGE1:
    # 第一階段沒有訓練 CTC，所以只取它的分類層機率
    val_scores = (stage2_val + stage1_scores["val"][:, STAGE1_COLUMNS]) / 2
    test_scores = (stage2_test + stage1_scores["test"][:, STAGE1_COLUMNS]) / 2
else:
    val_scores, test_scores = stage2_val, stage2_test

clips = [p.rsplit("_s", 1)[0] for p, _ in test_rows]
mask = eval_words(test_labels)
print(f"測試集 {len(test_rows)} 句，{int(mask.sum())} 個詞算進 mAP")
if ENSEMBLE_STAGE1:
    print(f"  mAP (只用第二階段)  {mean_ap(test_labels, stage2_test, mask):.3f}")
print(f"  mAP{' (兩階段平均)' if ENSEMBLE_STAGE1 else '               '}  {mean_ap(test_labels, test_scores, mask):.3f}")
print(f"  同則新聞內打亂 mAP  {mean_ap(test_labels, shuffle_within(test_scores, clips), mask):.3f}  (模型只認得新聞主題時的分數)")
print(f"  亂猜               {float(test_labels[:, mask].mean(0).mean()):.3f}")

stage2:  62%|██████▏   | 93/150 [40:23<24:45, 26.06s/it, best=0.310, loss=0.3837, val_mAP=0.304]  


stage2 完成：最佳 epoch 84，val mAP 0.310
測試集 2775 句，245 個詞算進 mAP
  mAP (只用第二階段)  0.288
  mAP (兩階段平均)  0.303
  同則新聞內打亂 mAP  0.052  (模型只認得新聞主題時的分數)
  亂猜               0.004


## 5. 評估與挑門檻
在驗證集上挑門檻，讓「有把握」的詞有 70% 是對的，再到測試集報告實際效果

In [13]:
def outputs(s, threshold):
    out = np.zeros_like(s, dtype=bool)
    for i, row in enumerate(s):
        idx = [j for j in np.argsort(-row)[:MAX_WORDS] if row[j] >= threshold]
        out[i, idx] = True
    return out


def evaluate_output(s, y, threshold, rows_mask=None):
    rows_mask = np.ones(len(s), dtype=bool) if rows_mask is None else rows_mask
    out, y = outputs(s[rows_mask], threshold), y[rows_mask]
    answered = out.any(1)
    hits = (out & (y > 0)).sum()
    return {
        "precision": hits / max(out.sum(), 1),
        "answered": answered.mean(),
        "any_hit": (out & (y > 0)).any(1)[answered].mean() if answered.any() else float("nan"),
        "recall": hits / max(y.sum(), 1),
        "words_per_answer": out.sum() / max(answered.sum(), 1),
    }


def pick_threshold(target):
    for t in np.arange(0.0, 1.0001, 0.005):
        if evaluate_output(val_scores, val_labels, t)["precision"] >= target:
            return float(t)
    return 1.0


seen = {t for _, t in train_rows}
new_rows = np.array([t not in seen for _, t in test_rows])
print(f"{'驗證集目標':<10}{'門檻':>6}{'測試精確率':>10}{'有回答':>8}{'至少答對1個':>12}{'召回率':>8}{'新句精確率':>10}")
for target in sorted({0.6, 0.65, 0.7, 0.75, 0.8, TARGET_PRECISION}):
    t = pick_threshold(target)
    r, rn = evaluate_output(test_scores, test_labels, t), evaluate_output(test_scores, test_labels, t, new_rows)
    mark = "  ← 使用" if target == TARGET_PRECISION else ""
    print(f"{target:<12.0%}{t:>6.3f}{r['precision']:>11.1%}{r['answered']:>9.1%}{r['any_hit']:>12.1%}"
          f"{r['recall']:>9.1%}{rn['precision']:>11.1%}{mark}")

THRESHOLD = pick_threshold(TARGET_PRECISION)
result = evaluate_output(test_scores, test_labels, THRESHOLD)
print(f"\n使用門檻 {THRESHOLD:.3f}：測試集精確率 {result['precision']:.1%}，"
      f"{result['answered']:.1%} 的句子有回答，平均每次輸出 {result['words_per_answer']:.1f} 個詞")
if result["precision"] < 0.6:
    print("[注意] 測試集精確率低於 60%：把 TARGET_PRECISION 調高，再從這一格重新執行")


def two_layers(s, threshold):
    """每句的 (有把握, 其他可能) 布林矩陣，跟 keyword_predict.predict_with_candidates 的規則相同。"""
    conf, other = np.zeros_like(s, dtype=bool), np.zeros_like(s, dtype=bool)
    for i, row in enumerate(s):
        order = np.argsort(-row)
        c = [j for j in order if row[j] >= threshold][:min(MAX_WORDS, SHOW_WORDS)]
        o = [j for j in order if j not in c][:max(0, SHOW_WORDS - len(c))]
        conf[i, c], other[i, o] = True, True
    return conf, other


conf, other = two_layers(test_scores, THRESHOLD)
truth = test_labels > 0
shown = conf | other
print(f"\n兩層輸出 (每句顯示 {SHOW_WORDS} 個詞)，測試集：")
print(f"  有把握    ：平均每句 {conf.sum(1).mean():.2f} 個詞，精確率 {(conf & truth).sum() / max(conf.sum(), 1):.1%}")
print(f"  其他可能  ：平均每句 {other.sum(1).mean():.2f} 個詞，精確率 {(other & truth).sum() / max(other.sum(), 1):.1%}")
print(f"  合計      ：每句顯示的 {SHOW_WORDS} 個詞裡平均答對 {(shown & truth).sum(1).mean():.2f} 個；"
      f"字幕中在詞彙清單裡的詞平均 {truth.sum(1).mean():.2f} 個 (這是能答對的上限)")
in_vocab = truth.sum(1)
print(f"  參考      ：字幕平均有 {np.mean([len(words_of(t)) for _, t in test_rows]):.1f} 個詞，"
      f"其中在詞彙清單裡的平均 {in_vocab.mean():.2f} 個；有 >= 3 個在清單裡的句子 {np.mean(in_vocab >= 3):.1%}")

config = {
    "keywords": KEYWORDS,
    "threshold": THRESHOLD,
    "max_words": MAX_WORDS,
    "show_words": SHOW_WORDS,
    "checkpoint": "stage2/best.pt",                                        # 相對於 OUT_DIR
    "ensemble": ({"checkpoint": "stage1/best.pt", "n_keywords": len(STAGE1_KEYWORDS), "columns": STAGE1_COLUMNS}
                 if ENSEMBLE_STAGE1 else None),
    "norm_file": Path("..", DATA_DIR, NORM_FILE).as_posix(),                # 相對於 OUT_DIR
    "feat_dim": FEAT_DIM,
    "use_face": USE_FACE,
    "frame_stride": FRAME_STRIDE,
    "max_frames": MAX_FRAMES,
    "model": {"d_model": D_MODEL, "n_heads": N_HEADS, "enc_layers": ENC_LAYERS, "ff_dim": FF_DIM, "dropout": DROPOUT},
    "expected_test": {**{k: round(float(v), 4) for k, v in result.items()},
                      "candidate_precision": round(float((other & truth).sum() / max(other.sum(), 1)), 4)},
}
(OUT_DIR / "demo_config.json").write_text(json.dumps(config, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"已存成 {OUT_DIR / 'demo_config.json'} (keyword_predict.py 會讀這個檔案)")

驗證集目標         門檻     測試精確率     有回答      至少答對1個     召回率     新句精確率
60%          0.585      57.8%    31.6%       58.5%    31.1%      37.6%
65%          0.635      62.1%    27.8%       61.6%    29.2%      39.9%
70%          0.690      67.4%    24.0%       65.6%    27.4%      43.2%  ← 使用
75%          0.725      72.0%    21.3%       68.4%    25.8%      46.1%
80%          0.765      76.6%    18.2%       73.1%    23.1%      48.7%

使用門檻 0.690：測試集精確率 67.4%，24.0% 的句子有回答，平均每次輸出 1.9 個詞

兩層輸出 (每句顯示 5 個詞)，測試集：
  有把握    ：平均每句 0.45 個詞，精確率 67.4%
  其他可能  ：平均每句 4.55 個詞，精確率 5.7%
  合計      ：每句顯示的 5 個詞裡平均答對 0.56 個；字幕中在詞彙清單裡的詞平均 1.11 個 (這是能答對的上限)
  參考      ：字幕平均有 4.7 個詞，其中在詞彙清單裡的平均 1.11 個；有 >= 3 個在清單裡的句子 12.8%
已存成 keyword_model\demo_config.json (keyword_predict.py 會讀這個檔案)


## 6. 示範
確認 keyword_predict.py 的結果和訓練時一致，並實際示範預測

In [14]:
import importlib

import keyword_predict

importlib.reload(keyword_predict)
predictor = keyword_predict.KeywordPredictor(OUT_DIR / "demo_config.json")

check = np.random.default_rng(0).choice(len(test_rows), min(20, len(test_rows)), replace=False)
diff = max(np.abs(predictor.scores(SEG_DIR / test_rows[i][0]) - test_scores[i]).max() for i in check)
print(f"{len(check)} 句的最大分數差距：{diff:.4f}" + ("  → 一致" if diff < 0.02 else "  → [注意] 差距偏大，請確認設定"))

20 句的最大分數差距：0.0004  → 一致


In [15]:
NPY = SEG_DIR / test_rows[0][0]          # ← 換成想測試的 .npy 路徑 (03segments 裡的檔案)

subtitles = {p: t for p, t in train_rows + val_rows + test_rows}
confident, others = predictor.predict_with_candidates(NPY)
key = Path(NPY).relative_to(SEG_DIR).as_posix() if Path(NPY).is_relative_to(SEG_DIR) else None
text = subtitles.get(key)
truth_words = words_of(text) if text else set()


def fmt(words):
    if not words:
        return "無"
    return "、".join(f"{w} ({p:.2f}){' ✔' if w in truth_words else (' ✘' if text else '')}" for w, p in words)


print(f"輸入：{NPY}")
if text:
    print(f"字幕：{text}")
print(f"有把握  ：{fmt(confident)}")
print(f"其他可能：{fmt(others)}")

輸入：..\Data Preparation\Phase5-Pause\output\03segments_face\1\4\000598_001348_s00.npy
字幕：台中有一名陳姓男子
有把握  ：無
其他可能：日本 (0.19) ✘、台灣 (0.12) ✘、疑似 (0.04) ✘、必須 (0.04) ✘、昨天 (0.03) ✘


In [16]:
SHOW = 15          # 隨機示範幾句測試資料 (至少有 1 個「有把握」的詞)
rng = np.random.default_rng(1)
picked = [i for i in rng.permutation(len(test_rows)) if test_scores[i].max() >= THRESHOLD][:SHOW]
stats = {"有把握": [0, 0], "其他可能": [0, 0]}
for i in picked:
    path, text = test_rows[i]
    truth_words = words_of(text)
    confident, others = predictor.predict_with_candidates(SEG_DIR / path)
    for name, words in (("有把握", confident), ("其他可能", others)):
        stats[name][0] += len(words)
        stats[name][1] += sum(w in truth_words for w, _ in words)
    mark = lambda words: "、".join(f"{w} ({p:.2f}){' ✔' if w in truth_words else ' ✘'}" for w, p in words) or "無"
    print(f"字幕：{text}\n  有把握  ：{mark(confident)}\n  其他可能：{mark(others)}")
print()
for name, (n, hit) in stats.items():
    print(f"這 {len(picked)} 句的「{name}」共 {n} 個詞，{hit} 個正確 ({hit / max(n, 1):.0%})")

字幕：王品集團宣布明年元旦開始
  有把握  ：宣布 (0.82) ✔
  其他可能：明年 (0.68) ✔、昨天 (0.50) ✘、公布 (0.48) ✘、最快 (0.24) ✘
字幕：立法院衛環委員會舉行公聽會
  有把握  ：委員會 (0.82) ✔
  其他可能：立法院 (0.54) ✔、衛福部 (0.32) ✘、支持 (0.22) ✘、修正 (0.22) ✘
字幕：關心天氣
  有把握  ：天氣 (1.00) ✔、關心 (1.00) ✔
  其他可能：今年 (0.22) ✘、升溫 (0.12) ✘、高溫 (0.07) ✘
字幕：這次首度有3家業者表示
  有把握  ：表示 (0.98) ✔、這次 (0.93) ✔
  其他可能：參與 (0.59) ✘、總統 (0.44) ✘、接受 (0.26) ✘
字幕：疫情趨緩加上邊境解封
  有把握  ：今天 (0.80) ✘、疫情 (0.70) ✔
  其他可能：台灣 (0.51) ✘、開放 (0.29) ✘、地區 (0.16) ✘
字幕：南部地區豪雨來襲
  有把握  ：今天 (0.77) ✘
  其他可能：南部 (0.53) ✔、影響 (0.48) ✘、台灣 (0.41) ✘、颱風 (0.36) ✘
字幕：新北市41處公有市場昨天起轉型成不塑市場
  有把握  ：昨天 (0.86) ✔
  其他可能：方案 (0.43) ✘、調漲 (0.17) ✘、執行 (0.14) ✘、發生 (0.14) ✘
字幕：根據統計
  有把握  ：統計 (0.88) ✔
  其他可能：內政部 (0.47) ✘、月底 (0.33) ✘、公布 (0.25) ✘、去年 (0.12) ✘
字幕：讀報時間來看
  有把握  ：來看 (1.00) ✔、讀報 (1.00) ✔、時間 (1.00) ✔
  其他可能：報導 (0.01) ✘、自由 (0.01) ✘
字幕：自由時報報導 台中地檢署昨天認定
  有把握  ：報導 (0.99) ✔、自由 (0.99) ✔、時報 (0.99) ✔
  其他可能：公布 (0.65) ✘、昨天 (0.36) ✔
字幕：立法院昨天針對總預算
  有把握  ：立法院 (0.92) ✔
  其他可能：通過 (0.36) ✘、今年 (0.20) ✘、昨天 (0.14) ✔、行政院 (0.13) ✘
字幕：全案昨天偵結
 